### 1. Reading a fixed-width file 

Convert it to a columnar analytics-friendly representation (input_df)

In [ ]:
from file_parser import parse, get_schema_from_copybook
from file_parser.constants import INPUT_PATH, INTERMEDIATE_OUTPUT_PATH, FORMULAS_PATH
from file_parser.parsers.polars import file_parser

In [2]:

COPYBOOK = """
       01  FILE-RECORD.
           05  FULL-NAME                  PIC X(50).
           05  YEAR                       PIC 9(4).
           05  AMOUNT                     PIC 9(09)V99.
"""

file_schema = get_schema_from_copybook(COPYBOOK)
input_df = parse(INPUT_PATH, INTERMEDIATE_OUTPUT_PATH, file_schema, file_parser)

input_df.head(10).collect()

FULL_NAME,YEAR,AMOUNT
str,i64,"decimal[11,2]"
"""RAHGTSYCLAFNAFROFPVAVSJEZJCCWQ…",6065,272956798.27
"""SSSBROGMHYSFIUBWVKBYPTFNXRDDUO…",1876,457025829.31
"""DLMZXHNEYXIRQEUOVOAIAZXWIBXZCN…",5539,773997527.93
"""OPATBBAINHNOTXPGMKCRJLXBRRBTVC…",2697,16276771.55
"""HVMLZ PTEI POUBPNXEZCFQSGDYGQQ…",5718,123732174.09
"""XJEWSQ RAWIRZDDCOHQTFRHNYWCLHA…",3441,329942252.06
"""SJPX UYGEVELEYVLSTGESKBMFYJWXG…",8379,520363024.25
"""YFBLOVSZTFLZYQRDYNSISOFRKUEPKW…",2414,43998735.51
"""GMWBLRFSNGRAUUCLEZNBGWMVS QZYQ…",3788,961344045.63


### 2. Validating the columnar representation of the file

Apply validations defined in the formulas and generate and enriched dataset with 1 additional column by validation formula.

This are the formulas used to validate the input file:

In [8]:
!cat ../formulas.txt

VALID_NAME: LEN({FULL_NAME}) == 9
VALID_YEAR: BETWEEN({YEAR}, 1900, 2026)
VALID_AMOUNT: BETWEEN({AMOUNT}, 0, 672581176.44)

In [3]:
from formula_engine import compute

In [4]:
result_df = compute(FORMULAS_PATH, input_df)
result_df.head(10).collect()

FULL_NAME,YEAR,AMOUNT,VALID_NAME,VALID_YEAR,VALID_AMOUNT
str,i64,"decimal[11,2]",bool,bool,bool
"""RAHGTSYCLAFNAFROFPVAVSJEZJCCWQ…",6065,272956798.27,false,false,true
"""SSSBROGMHYSFIUBWVKBYPTFNXRDDUO…",1876,457025829.31,false,false,true
"""DLMZXHNEYXIRQEUOVOAIAZXWIBXZCN…",5539,773997527.93,false,false,false
"""OPATBBAINHNOTXPGMKCRJLXBRRBTVC…",2697,16276771.55,false,false,true
"""HVMLZ PTEI POUBPNXEZCFQSGDYGQQ…",5718,123732174.09,false,false,true
"""XJEWSQ RAWIRZDDCOHQTFRHNYWCLHA…",3441,329942252.06,false,false,true
"""SJPX UYGEVELEYVLSTGESKBMFYJWXG…",8379,520363024.25,false,false,true
"""YFBLOVSZTFLZYQRDYNSISOFRKUEPKW…",2414,43998735.51,false,false,true
"""GMWBLRFSNGRAUUCLEZNBGWMVS QZYQ…",3788,961344045.63,false,false,false
